# Customer Segmentation using K-Means Clustering

## Capstone Project – Week 4

**Goal:** Segment mall customers into meaningful groups based on annual income and spending behavior using K-Means clustering.

### Problem Statement
A business has customer data but does not know which customers have similar spending behavior. This project groups customers into segments so the business can better understand different customer groups and design targeted marketing strategies.


## 1. Import Libraries
We use Pandas for data handling, Matplotlib/Seaborn for visualization, and Scikit-learn for K-Means and evaluation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

ModuleNotFoundError: No module named 'seaborn'

## 2. Load Dataset
Upload `Mall Customers.xlsx` to Colab or keep it in the same project folder.

In [ ]:
df = pd.read_excel('Mall Customers.xlsx')
df.head()

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.info()

## 3. Data Quality Check

In [ ]:
print('Missing values:')
print(df.isnull().sum())

print('\nDuplicate rows:', df.duplicated().sum())

## 4. Exploratory Data Analysis (EDA)
We inspect the distributions and relationship between annual income and spending score.

In [ ]:
df.describe(include='all')

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='Annual Income (k$)', y='Spending Score (1-100)')
plt.title('Annual Income vs Spending Score')
plt.show()

## 5. Select Features
For customer segmentation, we use **Annual Income** and **Spending Score** because these features directly represent customer purchasing behavior.

In [ ]:
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].copy()
X.head()

## 6. Elbow Method
We test different values of K and compare the Within-Cluster Sum of Squares (inertia). The elbow helps identify a suitable number of clusters.

In [ ]:
inertias = []
K_range = range(2, 11)

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X)
    inertias.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(list(K_range), inertias, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(list(K_range))
plt.show()

## 7. Silhouette Score
Silhouette Score helps evaluate how well-separated the clusters are. A higher score generally indicates better-defined clusters.

In [ ]:
silhouette_scores = {}

for k in K_range:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores[k] = score
    print(f'K={k}: Silhouette Score={score:.4f}')

plt.figure(figsize=(8,5))
plt.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Different K Values')
plt.xticks(list(K_range))
plt.show()

## 8. Train Final K-Means Model
For the final model, choose K after considering both the Elbow Method and Silhouette Score. The value below is set to 5 as a common starting point for this dataset; update it if your analysis supports another K.

In [ ]:
FINAL_K = 5

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

df.head()

## 9. Visualize Customer Clusters

In [ ]:
plt.figure(figsize=(9,6))
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Cluster',
    palette='viridis',
    s=80
)
centers = kmeans.cluster_centers_
plt.scatter(centers[:,0], centers[:,1], marker='X', s=250, label='Centroids')
plt.title('Customer Segments using K-Means')
plt.legend()
plt.show()

## 10. Analyze Each Cluster
We calculate the average income and spending score for each cluster. This allows us to give business-friendly interpretations to the cluster numbers.

In [ ]:
cluster_summary = df.groupby('Cluster')[['Annual Income (k$)', 'Spending Score (1-100)']].mean().round(2)
cluster_summary['Customer Count'] = df['Cluster'].value_counts().sort_index()
cluster_summary

### Interpretation
Do not assume that Cluster 0, 1, 2, etc. has a fixed meaning. Interpret each cluster from its average income and spending score. For example, a cluster with high income and high spending can be described as a high-value/high-spending segment.

## 11. Save Results
The final dataset now contains the assigned cluster for every customer.

In [ ]:
df.to_csv('customer_segments.csv', index=False)
print('Saved: customer_segments.csv')

## 12. Conclusion
K-Means clustering was used to divide customers into groups based on annual income and spending score. The Elbow Method and Silhouette Score were used to compare possible cluster counts. The final cluster analysis provides interpretable customer segments that can support targeted marketing decisions.